# Research Version 3 ? 00: Environment and persistence

Create an isolated CPU environment, capture package versions and verify Drive recovery. A successful local syntax check is not evidence of a successful hosted Colab run.

This notebook runs only in Colab and Google Drive. No Windows checkout, local datasets, local model files or replication-manifest file are required. Use the new experiment ID; do not mix this package with an older cpu-v3 archive.


# 00: Colab environment and shared Drive utilities

## 1. Purpose

Create the CPU environment and shared Drive runtime. No dataset acquisition or model training occurs.

## 2. Prerequisites and outputs

**Required inputs:** Nothing from earlier stages.

**This stage saves:** Shared controller, exact package freeze and a successful setup marker.

Required earlier notebook markers: none.

All runtime files belong to this Colab/Drive experiment; no external project files are read.

## 3. Configuration

Use the same experiment ID/folder throughout. All action switches default False. Enable Drive/bootstrap and RUN_STAGE deliberately; enable installation on a fresh runtime. Notebook 01 needs dataset download permission; notebook 08 needs pretrained download permission.

In [ ]:
EXPERIMENT_ID = "colab-cpu-v3-clean"
DRIVE_FOLDER = "ChainShield-Colab"
ENABLE_DRIVE = False
BOOTSTRAP_EXPERIMENT = False
ALLOW_PACKAGE_INSTALL = False
ALLOW_DATA_DOWNLOADS = False
ALLOW_PRETRAINED_DOWNLOADS = False
RUN_STAGE = False
SYNC_INTERVAL_SECONDS = 60
VERIFY_PERSISTENCE_FULL = False
VERIFY_CHECKPOINT_PAIRS = False
CLEAR_STALE_WRITER_LOCK = False
CONFIRM_NO_OTHER_WRITER = False
DOWNLOAD_REPORT_ZIP = False

# Isolated plotting workers must not inherit the Colab inline backend.
if RUN_STAGE:
    import os
    os.environ["MPLBACKEND"] = "Agg"


## 4. Readable stage implementation

The small cell command below registers visible module text in memory. It does not execute the module. This notebook supplies only the files listed here; other supporting modules are restored from earlier Colab stages.

In [ ]:
# This small cell registers the visible stage code; it writes no files.
REGISTERED_FILES = {}
def register_stage_file(line, body):
    name=line.strip()
    if not name or name.startswith("/") or "\\" in name or ":" in name or ".." in name.split("/"):
        raise ValueError("Use a safe relative module name")
    REGISTERED_FILES[name]=body.rstrip("\n")+"\n"
get_ipython().register_magic_function(register_stage_file,"cell","chainshield_file")


### .replication/controller.py


In [ ]:
%%chainshield_file .replication/controller.py

import os, sys, json, hashlib, shutil, subprocess, platform, time, threading, csv, uuid, re
from pathlib import Path
from datetime import datetime, timezone
from contextlib import contextmanager

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pending = path.with_name(path.name + ".incoming-" + uuid.uuid4().hex)
    pending.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(pending, path)

def safe_relative(value):
    value = str(value)
    p = Path(value)
    if p.is_absolute() or ".." in p.parts or "\\" in value or ":" in value or not p.parts:
        raise ValueError("Unsafe persisted relative path: " + value)
    return p



def expected_file_record(path, expected):
    path = Path(path)
    return path.is_file() and file_sha(path) == expected

class Replica:
    """Colab-owned runtime plus bounded mirror persistence; readable training modules supplied by this notebook."""
    def __init__(self, experiment_id, drive_base="ChainShield-Colab", allow_drive=False):
        if not re.fullmatch(r"[A-Za-z0-9_-]{1,64}", experiment_id):
            raise ValueError("Use an experiment ID without paths or spaces")
        if not allow_drive or not Path("/content").is_dir() or not sys.platform.startswith("linux"):
            raise RuntimeError("Enable Drive/bootstrap in a Linux Colab runtime")
        from google.colab import drive
        if not Path("/content/drive/MyDrive").is_dir():
            drive.mount("/content/drive")
        base=Path(drive_base)
        if base.is_absolute() or ".." in base.parts:
            raise ValueError("Drive folder must be relative to MyDrive")
        self.id=experiment_id
        self.root=Path("/content/chainshield_colab")/experiment_id
        self.env=Path("/content/chainshield_environments")/experiment_id
        self.python=self.env/"bin/python"
        self.archive=Path("/content/drive/MyDrive")/base/"experiments"/experiment_id
        self.mirror=self.archive/"workspace"
        self.control=self.archive/"control"
        self.run_dir=self.root/".replication/runs"
        self.owner=uuid.uuid4().hex
        self.sync_lock=threading.RLock()
        self._hash_cache={}
        self._active_process=None
        self.root.mkdir(parents=True,exist_ok=True)
        self.control.mkdir(parents=True,exist_ok=True)
        self.run_dir.mkdir(parents=True,exist_ok=True)
        (self.root/".tmp").mkdir(exist_ok=True)
        (self.root/"docs").mkdir(exist_ok=True)
        binding={"schema_version":3,"experiment_id":self.id,"controller_sha256":CONTROLLER_SHA,
                 "runtime_root":str(self.root),"device":"cpu","format":"focused-colab-v3"}
        path=self.control/"experiment.json"
        if path.exists() and read_json(path)!=binding:
            raise ValueError("Version 3 experiment controller/path binding changed; preserve the archive")
        if not path.exists():write_json(path,binding)
        index=self.control/"workspace-index.json"
        self.sources=read_json(index).get("source_hashes",{}) if index.exists() else {}
        self.restore()
        self.verify_source()
        print("Colab runtime:",self.root)
        print("Persistent archive:",self.archive)



    def verify_source(self):
        for name, expected in self.sources.items():
            if not expected_file_record(self.root/safe_relative(name),expected):
                raise ValueError("Stage-supplied implementation changed: "+name)

    def install_sources(self, sources):
        """Commit only this notebook's visible files; never replace a changed definition."""
        with self.writer():
            additions={}
            for name, text in sources.items():
                relative=safe_relative(name)
                digest=hashlib.sha256(text.encode("utf-8")).hexdigest()
                if name in self.sources and self.sources[name]!=digest:
                    raise ValueError("A previous stage supplied different code: "+name)
                target=self.root/relative
                if target.exists() and file_sha(target)!=digest:
                    raise ValueError("Conflicting runtime file; restart and restore: "+name)
                additions[name]=digest
            for name,text in sources.items():
                target=self.root/safe_relative(name)
                if not target.exists():
                    target.parent.mkdir(parents=True,exist_ok=True)
                    pending=target.with_name(target.name+".source-pending")
                    pending.write_text(text,encoding="utf-8")
                    os.replace(pending,target)
            self.sources.update(additions)
            self.sync()
        print("Stage implementation files committed:",len(sources))

    def require_stage(self, number):
        if not (self.control/"notebooks"/(number+".json")).is_file():
            raise RuntimeError("Complete Colab notebook "+number+" first")

    def stage_done(self, number):
        return (self.control/"notebooks"/(number+".json")).is_file()

    def finish_stage(self, number):
        with self.writer():
            self.verify_stage_outputs(number)
            count=self.sync()
            write_json(self.control/"notebooks"/(number+".json"),
                       {"notebook":number,"completed_at_utc":utcnow(),
                        "controller_sha256":CONTROLLER_SHA,"indexed_files":count})
        print("Notebook",number,"completed and persisted;",count,"indexed files")


    def verify_stage_outputs(self, number):
        terminals={
            "01":["data/manifests/"+n+".json" for n in ("dataco","india_nepal","crisismmd")],
            "02":["data/processed/"+n+"/quality.json" for n in ("dataco","india_nepal","crisismmd")]+["docs/dataset-readiness.md"],
            "03":["artifacts/baselines/shipment_summary.json"],
            "04":["artifacts/baselines/text_summary.json","artifacts/baselines/baseline_summary.json"],
            "05":["artifacts/baselines/robustness/summary.json"],
            "06":["artifacts/generalization/summary.json"],
            "07":["artifacts/neural/main/screening/main/completion.json"],
            "08":["artifacts/neural/main/"+n+"/main/completion.json" for n in ("disaster_type","humanitarian")]+["artifacts/neural/pretrained/distilbert/manifest.json"],
            "09":["artifacts/neural/summary.json"],
            "10":[".replication/reports/comparison.json"],
            "11":[".replication/reports/inference.json"],
            "12":[".replication/reports/errors.json"],
            "14":[".replication/reports/research-report"+suffix for suffix in (".md",".html",".pdf",".docx")]+[".replication/reports/research-inventory.json"]}
        if number=="02D":
            self.require("data/processed/research-v2/paired_v2/completion.json","docs/dataset-readiness.md")
            committed=self.collect_committed()
            if "data/processed/research-v2/paired_v2/completion.json" not in committed:raise RuntimeError("Handoff is not committed")
            return
        if number in ("02A","02B","02C"):
            dataset={"02A":"dataco","02B":"india_nepal","02C":"crisismmd"}[number]
            required=["data/processed/"+dataset+"/quality.json","data/processed/research-v2/dataset_stages/"+dataset+"/completion.json"]
            for variant in (("control","url_mentions") if dataset=="crisismmd" else ("control",)):
                required.append("artifacts/research-v2/eda/"+dataset+"_"+variant+"/visualization-completion-v2.json")

            self.require(*required)
            committed=self.collect_committed()
            if any(name not in committed for name in required):raise RuntimeError("Dataset stage evidence is not committed")
            return
        if number=="00":
            if not (self.env/"environment-complete.json").is_file() or not (self.control/"requirements-colab-resolved.txt").is_file():
                raise RuntimeError("Environment setup has not completed")
        elif number=="13":
            path=self.control/"stages/13-verification.json"
            if not path.is_file() or read_json(path).get("exit_code")!=0:
                raise RuntimeError("Correctness checks have not passed")
        elif number in terminals:
            extras={"02":["data/processed/research-v2/paired_v2/completion.json"]+["artifacts/research-v2/eda/paired_v2_"+v+"/visualization-completion-v2.json" for v in ("control","url_mentions")],
                    "03":["artifacts/research-v2/search/dataco/summary.json"],
                    "04":["artifacts/research-v2/search/"+d+"/summary.json" for d in ("india_nepal","crisismmd")],
                    "10":["artifacts/research-v2/evaluation/completion.json"]}
            self.require(*(terminals[number]+extras.get(number,[])))
            committed=self.collect_committed()
            for name in terminals[number]:
                if name not in committed:
                    raise RuntimeError("Stage terminal output is not committed: "+name)
        else:
            raise ValueError("Unknown notebook number")
        checks={"03":("artifacts/baselines/shipment_summary.json","results",4),
                "04":("artifacts/baselines/baseline_summary.json","results",9),
                "05":("artifacts/baselines/robustness/summary.json","results",5),
                "06":("artifacts/generalization/summary.json","results",20)}
        if number in checks:
            path,key,count=checks[number]
            if len(read_json(self.root/path)[key])!=count:
                raise RuntimeError("Stage experiment count is incomplete")
        if number=="09":
            item=read_json(self.root/"artifacts/neural/summary.json")
            if not item.get("complete") or item.get("completed_jobs")!=19 or item.get("test_evaluated") is not False:
                raise RuntimeError("Nineteen development-only neural jobs are required")
        if number=="10":
            if not read_json(self.root/".replication/reports/comparison.json").get("complete"):
                raise RuntimeError("Research comparison is incomplete")

    @contextmanager
    def writer(self):
        lock = self.control / "writer-lock.json"
        record = {"owner": self.owner, "created_at_utc": utcnow(), "runtime_root": str(self.root)}
        try:
            with lock.open("x", encoding="utf-8") as stream:
                json.dump(record, stream)
        except FileExistsError:
            raise RuntimeError("Writer lock exists. Stop any other writer; inspect the recovery cell before clearing a stale lock.")
        try:
            yield
        finally:
            if lock.exists() and read_json(lock).get("owner") == self.owner:
                lock.unlink()

    def clear_stale_lock(self, acknowledge_no_other_writer=False):
        if not acknowledge_no_other_writer:
            raise PermissionError("Explicitly confirm no other Colab session is writing this experiment")
        lock = self.control / "writer-lock.json"
        if lock.exists():
            print("Removing acknowledged stale managed lock:", read_json(lock))
            lock.unlink()

    @staticmethod
    def checked_setup_command(command):
        result = subprocess.run(command, capture_output=True, text=True)
        if result.stdout:
            print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
        if result.stderr:
            print(result.stderr, end="" if result.stderr.endswith("\n") else "\n")
        if result.returncode:
            raise RuntimeError("Colab environment setup failed (exit " + str(result.returncode)
                               + "): " + " ".join(map(str, command)) + "\n"
                               + result.stdout + result.stderr)

    def environment(self, allow_install=False):
        runtime_record = self.control / "environment-runtime.json"
        if runtime_record.exists() and read_json(runtime_record)["python"] != platform.python_version():
            raise ValueError("Python version changed; old results remain readable but training needs a compatible runtime or new experiment")
        frozen = self.control / "requirements-colab-resolved.txt"
        marker = self.env / "environment-complete.json"
        if not marker.exists():
            if not allow_install:
                raise PermissionError("Run notebook 00 with ALLOW_PACKAGE_INSTALL=True")
            self.env.parent.mkdir(parents=True, exist_ok=True)
            # Some Colab Python builds provide venv but omit ensurepip.
            # Host pip targets the isolated interpreter; system packages are not changed.
            help_result = subprocess.run([sys.executable, "-m", "pip", "--help"],
                                         capture_output=True, text=True)
            if help_result.returncode or "--python" not in help_result.stdout:
                raise RuntimeError("Colab host pip with --python support is required. No system packages were changed.\n"
                                   + help_result.stdout + help_result.stderr)
            self.checked_setup_command([sys.executable, "-m", "venv", "--without-pip", str(self.env)])
            self.checked_setup_command([sys.executable, "-m", "pip", "--python",
                                        str(self.python), "install", "pip"])
            command = [str(self.python), "-m", "pip"]
            if frozen.exists():
                subprocess.run(command + ["install", "-r", str(frozen), "--extra-index-url",
                                         "https://download.pytorch.org/whl/cpu"], check=True)
            else:
                subprocess.run(command + ["install", "torch>=2.5,<3", "--index-url",
                                         "https://download.pytorch.org/whl/cpu"], check=True)
                packages = ["numpy>=1.26,<3", "pandas>=2.2,<4", "scikit-learn>=1.5,<2",
                            "xgboost>=2.1,<4", "joblib>=1.4,<2", "threadpoolctl>=3,<4",
                            "transformers>=5,<6", "huggingface-hub>=0.26,<3", "safetensors>=0.4,<1", "matplotlib>=3.9,<4", "python-docx>=1.1,<2"]
                subprocess.run(command + ["install"] + packages, check=True)
            subprocess.run(command + ["check"], check=True)
            resolved = subprocess.check_output(command + ["freeze"], text=True)
            if "@ file:" in resolved or "-e " in resolved:
                raise ValueError("Nonportable package source in environment; inspect before recording")
            if not frozen.exists():
                frozen.write_text(resolved, encoding="utf-8")
            else:
                # Compare package identities rather than incidental output ordering.
                if set(resolved.splitlines()) != set(frozen.read_text().splitlines()):
                    raise ValueError("Restored package resolution differs; do not resume this experiment")
            write_json(runtime_record, {"python": platform.python_version(), "platform": platform.platform()})
            write_json(marker, {"python": platform.python_version(), "freeze_sha256": file_sha(frozen)})
        if not frozen.exists() or read_json(marker)["freeze_sha256"] != file_sha(frozen):
            raise ValueError("Environment record is missing or changed")
        current = subprocess.check_output([str(self.python), "-m", "pip", "freeze"], text=True)
        if set(current.splitlines()) != set(frozen.read_text().splitlines()):
            raise ValueError("Installed packages changed after freeze; use a new experiment")
        link = self.root / ".venv"
        if not link.exists():
            link.symlink_to(self.env, target_is_directory=True)
        if link.resolve() != self.env.resolve():
            raise ValueError("Project-local environment points elsewhere")
        os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
        return self.python


    def restore(self):
        index_path = self.control / "workspace-index.json"
        if not index_path.exists():
            return
        index = read_json(index_path)
        if index.get("implementation_sha256") != IMPLEMENTATION_SHA256:
            raise ValueError("Persisted archive belongs to another notebook implementation")
        files = index.get("files", {})
        for name, record in files.items():
            relative = safe_relative(name)
            if any(part in {".venv", ".tmp", "__pycache__"} for part in relative.parts):
                raise ValueError("Unexpected runtime-only path in archive")
            source, target = self.mirror / relative, self.root / relative
            if target.exists():
                if file_sha(target) != record["sha256"]:
                    raise ValueError("Runtime differs from persisted evidence; restart runtime before recovery: " + name)
                continue
            self.copy_verified(source, target, record["sha256"])
        # Completed jobs need no optimizer state restored. Unfinished jobs restore only valid slot pairs.
        checkpoint_root = self.mirror / "artifacts/neural"
        pairs = {}
        if checkpoint_root.exists():
            for marker in checkpoint_root.rglob("checkpoint-*.json"):
                relative = marker.relative_to(self.mirror)
                complete = (relative.parent / "completion.json").as_posix()
                if complete in files:
                    continue
                pairs.setdefault(relative.parent, []).append(marker)
        for parent, markers in pairs.items():
            valid = 0
            for marker in markers:
                record = read_json(marker)
                weights = marker.with_suffix(".pt")
                if not expected_file_record(weights, record["sha256"]):
                    continue
                self.copy_verified(weights, self.root / weights.relative_to(self.mirror), record["sha256"])
                self.copy_verified(marker, self.root / marker.relative_to(self.mirror), file_sha(marker))
                valid += 1
            if not valid:
                raise ValueError("No valid persisted checkpoint pair for " + str(parent))
        print("Restored verified persisted files:", len(files), "; unfinished jobs with checkpoint records:", len(pairs))

    def copy_verified(self, source, target, expected):
        source, target = Path(source), Path(target)
        if target.exists() and file_sha(target) == expected:
            return
        target.parent.mkdir(parents=True, exist_ok=True)
        pending = target.with_name(target.name + ".incoming-" + uuid.uuid4().hex)
        h = hashlib.sha256()
        try:
            with source.open("rb") as src, pending.open("xb") as dst:
                for block in iter(lambda: src.read(1024 * 1024), b""):
                    h.update(block)
                    dst.write(block)
                dst.flush()
                os.fsync(dst.fileno())
            if h.hexdigest() != expected or file_sha(pending) != expected:
                raise ValueError("Source changed during copy or persisted copy failed: " + str(source))
            os.replace(pending, target)
        finally:
            if pending.exists():
                pending.unlink()

    def cached_sha(self, path):
        path = Path(path)
        stat = path.stat()
        signature = (stat.st_size, stat.st_mtime_ns)
        previous = self._hash_cache.get(str(path))
        if previous and previous[0] == signature:
            return previous[1]
        digest = file_sha(path)
        self._hash_cache[str(path)] = (signature, digest)
        return digest

    def collect_committed(self):
        selected = {}
        def add(path, expected=None):
            path = Path(path)
            if not path.is_file():
                raise FileNotFoundError(path)
            name = path.relative_to(self.root).as_posix()
            safe_relative(name)
            actual = self.cached_sha(path)
            if expected is not None and actual != expected:
                raise ValueError("Committed source failed integrity: " + name)
            selected[name] = {"sha256": actual, "bytes": path.stat().st_size}
        def optional(path):
            if Path(path).is_file():
                add(path)
        def committed_bundle(marker):
            record = read_json(marker)
            for name, expected in record["file_sha256"].items():
                safe_relative(name)
                add(marker.parent / name, expected)
            add(marker)
        for name, expected in self.sources.items():
            add(self.root / name, expected)
        for manifest in (self.root / "data/manifests").glob("*.json"):
            data = read_json(manifest)
            if data.get("status") != "downloaded_and_integrity_verified":
                continue
            for item in data["files"]:
                add(self.root / safe_relative(item["file"]), item["sha256"])
            add(manifest)
        quality_paths = [self.root / f"data/processed/{name}/quality.json" for name in ("dataco", "india_nepal", "crisismmd")]
        for quality in quality_paths:
            if quality.is_file() and (self.root/"docs"/("dataset-readiness-"+quality.parent.name+".md")).is_file():
                record=read_json(quality)
                for name,expected in record["output_sha256"].items():add(quality.parent/safe_relative(name),expected)
                add(quality)
        preparation_report = self.root / "docs/dataset-readiness.md"
        preparation_complete = all(p.is_file() for p in quality_paths) and preparation_report.is_file()
        if preparation_complete:
            preparation_complete = preparation_report.read_text(encoding="utf-8").rstrip().endswith("Model training must fit transformations only on training data; final test partitions stay sealed.")
        if preparation_complete:
            for quality in quality_paths:
                data = read_json(quality)
                for name, expected in data["output_sha256"].items():
                    add(quality.parent / safe_relative(name), expected)
                add(quality)
        baseline_root=self.root/"artifacts/baselines"
        for marker in baseline_root.glob("**/completion.json"):
            committed_bundle(marker)
        for name in ("shipment_summary.json","text_summary.json"):
            path=baseline_root/name
            if path.exists():
                expected=4 if name.startswith("shipment") else 5
                results=read_json(path)["results"]
                if len(results)!=expected:
                    raise ValueError("Incomplete baseline stage summary")
                for item in results:
                    folder=Path(item["artifact"])
                    folder.resolve().relative_to(self.root.resolve())
                    committed_bundle(folder/"completion.json")
                    if read_json(folder/"metadata.json")!=item["metadata"] or read_json(folder/"metrics.json")!=item["metrics"]:
                        raise ValueError("Baseline stage summary differs from committed artifacts")
                add(path)
        baseline_path = self.root / "artifacts/baselines/baseline_summary.json"
        if baseline_path.exists():
            baseline = read_json(baseline_path)
            if len(baseline["results"]) != 9:
                raise ValueError("Unified replication baseline stage must contain all nine tasks/models")
            for item in baseline["results"]:
                directory = Path(item["artifact"])
                directory.resolve().relative_to(self.root.resolve())
                if read_json(directory / "metadata.json") != item["metadata"]:
                    raise ValueError("Baseline metadata differs from summary")
                if read_json(directory / "metrics.json") != item["metrics"]:
                    raise ValueError("Baseline metrics differ from summary")
                add(directory / "model.joblib", item["metadata"]["artifact_sha256"])
                for name, expected in item["metadata"]["prediction_files"].items():
                    add(directory / name, expected)
                add(directory / "metadata.json")
                add(directory / "metrics.json")
            add(baseline_path)
        robust = self.root / "artifacts/baselines/robustness"
        if (robust / "summary.json").is_file():
            for audit in robust.glob("*/*/audit.json"):
                data = read_json(audit)
                for name, expected in data["output_sha256"].items():
                    add(audit.parent / safe_relative(name), expected)
                add(audit)
            add(robust / "summary.json")
        general = self.root / "artifacts/generalization"
        optional(general / "protocol.json")
        for marker in general.glob("**/completion.json"):
            committed_bundle(marker)
        optional(general / "summary.json")
        neural = self.root / "artifacts/neural"
        optional(neural / "run-binding.json")
        assets = neural / "pretrained/distilbert"
        if (assets / "manifest.json").is_file():
            data = read_json(assets / "manifest.json")
            for item in data["files"]:
                add(assets / safe_relative(item["file"]), item["sha256"])
            add(assets / "manifest.json")
        for job in neural.glob("**/job.json"):
            add(job)
        for marker in neural.glob("**/completion.json"):
            committed_bundle(marker)
        optional(neural / "summary.json")
        if self._active_process is None or self._active_process.poll() is not None:
            for report in (self.root / "docs").glob("*"):
                if report.is_file():
                    add(report)
        for base in [self.run_dir, self.root / ".replication/reports"]:
            if base.exists():
                for path in base.rglob("*"):
                    if path.is_file() and ".incoming-" not in path.name:
                        add(path)
        revised_paths=[self.root/"data/processed/research-v2/paired_v2"]+list((self.root/"data/processed/research-v2/dataset_stages").glob("*"))
        for revised in revised_paths:
            if (revised/"completion.json").exists():
                marker=read_json(revised/"completion.json")
                add(revised/"preparation-manifest.json",marker["manifest_sha256"])
                record=read_json(revised/"preparation-manifest.json")
                for name,expected in record["file_sha256"].items():add(revised/safe_relative(name),expected)
                add(revised/"completion.json")
        research=self.root/"artifacts/research-v2"
        token=research/"tokenizer/distilbert"
        if (token/"manifest.json").exists():
            for item in read_json(token/"manifest.json")["files"]:add(token/safe_relative(item["file"]),item["sha256"])
            add(token/"manifest.json")
        for marker in (research/"eda").glob("*/completion.json"):committed_bundle(marker)
        for marker in (research/"eda").glob("*/visualization-completion-v2.json"):committed_bundle(marker)
        for path in (research/"search").glob("**/*.json"):add(path)
        for marker in (research/"selected").glob("**/completion.json"):committed_bundle(marker)
        for marker in (research/"evaluation").glob("completion.json"):committed_bundle(marker)
        return selected

    def sync(self):
        with self.sync_lock:
            self.verify_source()
            latest = self.control / "workspace-index.json"
            old = read_json(latest).get("files", {}) if latest.exists() else {}
            selected = self.collect_committed()
            volatile = lambda name: name.startswith(".replication/") or name in {
                "artifacts/neural/summary.json", "docs/neural-experiments.md"}
            merged = dict(old)
            # Completion records are copied last; a committed index is published after all verified bytes.
            names = sorted(selected, key=lambda n: (n.endswith("completion.json"), n))
            for name in names:
                record = selected[name]
                if name in old and old[name]["sha256"] != record["sha256"] and not volatile(name):
                    raise ValueError("Attempted alteration of persisted immutable evidence: " + name)
                if name in old and old[name] == record:
                    continue
                try:
                    self.copy_verified(self.root / name, self.mirror / name, record["sha256"])
                except ValueError:
                    if volatile(name):
                        continue
                    raise
                merged[name] = record
            # Two rotating recovery slots, marker written last. Earlier slot remains a fallback.
            neural = self.root / "artifacts/neural"
            for marker in neural.glob("**/checkpoint-*.json"):
                record = read_json(marker)
                weights = marker.with_suffix(".pt")
                relative = marker.relative_to(self.root)
                destination = self.mirror / relative
                if destination.exists():
                    previous = read_json(destination)
                    if previous.get("generation") == record["generation"] and previous.get("sha256") == record["sha256"]:
                        continue
                if not expected_file_record(weights, record["sha256"]):
                    continue  # Trainer may be between replacing weights and committing its marker.
                print("[persistence] saving checkpoint", relative.as_posix(), "generation", record["generation"], flush=True)
                self.copy_verified(weights, self.mirror / weights.relative_to(self.root), record["sha256"])
                write_json(destination, record)
            write_json(latest, {"schema_version": 1, "implementation_sha256": IMPLEMENTATION_SHA256,
                                "updated_at_utc": utcnow(), "files": merged, "source_hashes": dict(self.sources)})
            return len(merged)

    def require(self, *relative_paths):
        missing = [name for name in relative_paths if not (self.root / name).exists()]
        if missing:
            raise FileNotFoundError("Missing persisted prerequisites; run the preceding notebooks: " + ", ".join(missing))

    def verify_archive(self, include_checkpoints=False):
        index = read_json(self.control / "workspace-index.json")
        for name, item in index["files"].items():
            if not expected_file_record(self.mirror / safe_relative(name), item["sha256"]):
                raise ValueError("Persisted evidence mismatch: " + name)
        checkpoint_count = 0
        if include_checkpoints:
            for marker in (self.mirror / "artifacts/neural").glob("**/checkpoint-*.json"):
                if not expected_file_record(marker.with_suffix(".pt"), read_json(marker)["sha256"]):
                    raise ValueError("Persisted checkpoint mismatch: " + str(marker))
                checkpoint_count += 1
        return {"verified_files": len(index["files"]), "verified_checkpoint_pairs": checkpoint_count}



    def resource_sample(self, pid):
        stat = Path(f"/proc/{pid}/stat").read_text()
        fields = stat[stat.rfind(")") + 2:].split()
        ticks = os.sysconf("SC_CLK_TCK")
        cpu_seconds = (int(fields[11]) + int(fields[12])) / ticks
        status = {}
        for line in Path(f"/proc/{pid}/status").read_text().splitlines():
            if line.startswith(("VmRSS:", "VmHWM:")):
                key, value = line.split(":", 1)
                status[key] = int(value.split()[0]) * 1024
        return {"timestamp_utc": utcnow(), "pid": pid, "cpu_seconds": cpu_seconds,
                "working_set_bytes": status.get("VmRSS", 0),
                "lifetime_peak_working_set_bytes": status.get("VmHWM", 0)}

    def execute(self, stage, arguments, sync_interval=60, allow_install=False):
        self.environment(allow_install=allow_install)
        self.verify_source()
        run_id = stage + "-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
        log_path = self.run_dir / (run_id + ".log")
        csv_path = self.run_dir / (run_id + "-resources.csv")
        status_path = self.run_dir / (run_id + "-status.json")
        resource_path = self.run_dir / (run_id + "-resource-summary.json")
        env = os.environ.copy()
        env.update(CUDA_VISIBLE_DEVICES="-1", PYTHONDONTWRITEBYTECODE="1",
                   HF_HUB_DISABLE_TELEMETRY="1", TOKENIZERS_PARALLELISM="false",
                   TEMP=str(self.root / ".tmp"), TMP=str(self.root / ".tmp"),
                   TMPDIR=str(self.root / ".tmp"))
        command = [str(self.root / ".venv/bin/python"), "-B"] + list(arguments)
        stop = threading.Event()
        resource_errors, sync_errors = [], []
        with self.writer():
            process = subprocess.Popen(command, cwd=self.root, env=env, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                                       errors="replace", bufsize=1)
            self._active_process = process
            state = {"stage": stage, "pid": process.pid, "running": True,
                     "started_at_utc": utcnow(), "progress": "Starting"}
            write_json(status_path, state)
            def monitor():
                previous = None
                peak_cpu, peak_ram, samples = 0.0, 0, 0
                with csv_path.open("x", encoding="utf-8", newline="") as stream:
                    writer = None
                    while not stop.is_set():
                        try:
                            row = self.resource_sample(process.pid)
                            current = time.monotonic()
                            cpu = 0.0
                            if previous is not None:
                                cpu = 100 * (row["cpu_seconds"] - previous[1]) / (current - previous[0]) / (os.cpu_count() or 1)
                            previous = (current, row["cpu_seconds"])
                            row["cpu_percent_normalized_logical_cpus"] = max(0.0, cpu)
                            if writer is None:
                                writer = csv.DictWriter(stream, fieldnames=list(row))
                                writer.writeheader()
                            writer.writerow(row)
                            stream.flush()
                            peak_cpu = max(peak_cpu, row["cpu_percent_normalized_logical_cpus"])
                            peak_ram = max(peak_ram, row["lifetime_peak_working_set_bytes"])
                            samples += 1
                            write_json(resource_path, {
                                "pid": process.pid, "samples": samples,
                                "updated_at_utc": row["timestamp_utc"],
                                "logical_processors": os.cpu_count(),
                                "sampled_peak_cpu_percent": peak_cpu,
                                "observed_peak_working_set_bytes": peak_ram,
                                "scope": "Actual Python worker; normalized CPU; sampled until process exit, not the full project lifetime"})
                        except FileNotFoundError:
                            break
                        except Exception as error:
                            resource_errors.append(str(error))
                            break
                        stop.wait(2)
            def persist():
                while not stop.wait(max(15, int(sync_interval))):
                    try:
                        count = self.sync()
                        print(f"[persistence] {count} indexed files; closed checkpoint pairs synchronized", flush=True)
                    except Exception as error:
                        sync_errors.append(str(error))
                        print("[persistence warning]", error, flush=True)
            resource_thread = threading.Thread(target=monitor, daemon=True)
            sync_thread = threading.Thread(target=persist, daemon=True)
            resource_thread.start()
            sync_thread.start()
            interrupted = False
            try:
                with log_path.open("x", encoding="utf-8", buffering=1) as log:
                    for line in process.stdout:
                        log.write(utcnow() + " " + line)
                        print(line, end="")
                        state["progress"] = line.strip()
                        state["updated_at_utc"] = utcnow()
                        write_json(status_path, state)
                exit_code = process.wait()
            except BaseException:
                interrupted = True
                if process.poll() is None:
                    process.terminate()
                    try:
                        process.wait(timeout=20)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        process.wait()
                raise
            finally:
                stop.set()
                resource_thread.join(timeout=10)
                sync_thread.join()
                state.update(running=False, exit_code=process.poll(),
                             interrupted=interrupted, finished_at_utc=utcnow(),
                             resource_errors=resource_errors,
                             earlier_persistence_warnings=sync_errors)
                write_json(status_path, state)
                # A final failure to persist is surfaced; do not report durable success.
                count = self.sync()
                print("Final persistence check:", count, "indexed files")
            if exit_code:
                raise RuntimeError(f"Stage failed ({exit_code}). Inspect {log_path}. Restart the runtime to restore only committed evidence; do not delete archive files.")
            write_json(self.control / "stages" / (stage + ".json"), {
                "stage": stage, "completed_at_utc": utcnow(), "run_id": run_id,
                "implementation_sha256": IMPLEMENTATION_SHA256, "exit_code": 0})
            print("Stage completed and persisted:", stage)

    def inventory(self):
        result = {"baselines": [], "generalization": [], "neural": []}
        base = self.root / "artifacts/baselines/baseline_summary.json"
        if base.exists():
            for item in read_json(base)["results"]:
                m, d = item["metrics"], item["metadata"]
                result["baselines"].append({
                    "dataset": d["dataset"], "task": d["task"],
                    "model": d.get("model_name"), "macro_f1": m["macro_f1"],
                    "accuracy": m["accuracy"], "roc_auc": m.get("roc_auc"),
                    "brier_score": m.get("brier_score")})
        for group in ["generalization", "neural"]:
            for marker in sorted((self.root / "artifacts" / group).glob("**/completion.json")):
                if not all(expected_file_record(marker.parent / name, digest)
                           for name, digest in read_json(marker)["file_sha256"].items()):
                    raise ValueError("Completion marker integrity mismatch: " + str(marker))
                m, d = read_json(marker.parent / "metrics.json"), read_json(marker.parent / "metadata.json")
                result[group].append({"dataset": d.get("dataset"), "task": d.get("task"),
                                      "model": d.get("model", d.get("model_name")),
                                      "kind": d.get("kind"), "name": d.get("name", marker.parent.name),
                                      "macro_f1": m["macro_f1"], "accuracy": m["accuracy"],
                                      "artifact": marker.parent.relative_to(self.root).as_posix()})
        result["counts"] = {key: len(value) for key, value in result.items()}
        result["test_evaluated"] = False
        return result


### src/chainshield/__init__.py


In [ ]:
%%chainshield_file src/chainshield/__init__.py
"""ChainShield AI: academic decision-support components."""

__version__ = "0.1.0"


### src/chainshield/data/__init__.py


In [ ]:
%%chainshield_file src/chainshield/data/__init__.py
"""Read-only dataset inspection and provenance helpers."""


### src/chainshield/data/audit.py


In [ ]:
%%chainshield_file src/chainshield/data/audit.py
"""Read-only structural audits; no inferred label mappings or split changes."""

from __future__ import annotations

from collections import Counter
import csv
import json
from pathlib import Path
from typing import Any, Iterable


def _audit_records(
    records: Iterable[dict[str, Any]],
    *,
    id_column: str | None,
    text_column: str | None,
    label_columns: tuple[str, ...],
    required_columns: tuple[str, ...],
    missing_tokens: tuple[str, ...],
) -> dict[str, Any]:
    missing_values = {value.casefold() for value in missing_tokens}
    columns: set[str] = set()
    missing_by_column: Counter[str] = Counter()
    absent_by_column: Counter[str] = Counter()
    labels = {name: Counter() for name in label_columns}
    identifiers: Counter[str] = Counter()
    texts: Counter[str] = Counter()
    malformed_rows = 0
    count = 0
    checks = set(required_columns) | set(label_columns)
    checks.update(name for name in (id_column, text_column) if name)

    def is_missing(value: Any) -> bool:
        return value is None or (
            isinstance(value, str) and value.strip().casefold() in missing_values
        )

    for row in records:
        if not isinstance(row, dict):
            raise ValueError("Every record must be an object with named fields")
        count += 1
        row_columns = {str(key) for key in row if key is not None}
        columns.update(row_columns)
        if None in row:
            malformed_rows += 1  # CSV row contained more fields than its header.
        for name in checks - row_columns:
            absent_by_column[name] += 1
        for name in row_columns:
            if is_missing(row[name]):
                missing_by_column[name] += 1
        for name, counter in labels.items():
            value = row.get(name)
            if not is_missing(value):
                counter[str(value).strip()] += 1
        if id_column and not is_missing(row.get(id_column)):
            identifiers[str(row[id_column]).strip()] += 1
        if text_column and not is_missing(row.get(text_column)):
            # Exact normalized-text candidates only, not semantic deduplication.
            key = " ".join(str(row[text_column]).casefold().split())
            texts[key] += 1

    return {
        "row_count": count,
        "columns": sorted(columns),
        "missing_requested_columns": sorted(checks - columns),
        "requested_field_absent_rows": dict(sorted(absent_by_column.items())),
        "missing_values_in_present_columns": dict(sorted(missing_by_column.items())),
        "malformed_csv_rows": malformed_rows,
        "label_counts": {name: dict(sorted(values.items())) for name, values in labels.items()},
        "duplicate_id_groups": sum(value > 1 for value in identifiers.values()),
        "duplicate_id_excess_rows": sum(value - 1 for value in identifiers.values()),
        "duplicate_normalized_text_groups": sum(value > 1 for value in texts.values()),
        "duplicate_normalized_text_excess_rows": sum(value - 1 for value in texts.values()),
        "limitations": [
            "No source text or example identifiers are included in this summary.",
            "Duplicate counters use memory proportional to distinct identifiers and texts.",
            "No cross-file split, near-duplicate, temporal, or label-conflict audit is performed.",
            "Labels are observed strings, not validated task semantics.",
        ],
    }


def audit_csv(
    path: str | Path,
    *,
    encoding: str = "utf-8-sig",
    delimiter: str = ",",
    id_column: str | None = None,
    text_column: str | None = None,
    label_columns: tuple[str, ...] = (),
    required_columns: tuple[str, ...] = (),
    missing_tokens: tuple[str, ...] = ("", "null", "none", "nan", "n/a"),
) -> dict[str, Any]:
    """Stream CSV/TSV rows and return a structural summary without writing files.

    Encoding and delimiter are explicit; no lossy decoding fallback is applied.
    Duplicate headers are rejected because DictReader would hide their values.
    """
    with Path(path).open("r", encoding=encoding, newline="") as handle:
        reader = csv.DictReader(handle, delimiter=delimiter)
        if not reader.fieldnames:
            raise ValueError("CSV requires a header")
        if len(reader.fieldnames) != len(set(reader.fieldnames)):
            raise ValueError("Duplicate CSV headers require explicit source reconciliation")
        return _audit_records(
            reader, id_column=id_column, text_column=text_column,
            label_columns=label_columns, required_columns=required_columns,
            missing_tokens=missing_tokens,
        )


def audit_json(
    path: str | Path,
    *,
    records_key: str | None = None,
    encoding: str = "utf-8-sig",
    id_column: str | None = None,
    text_column: str | None = None,
    label_columns: tuple[str, ...] = (),
    required_columns: tuple[str, ...] = (),
    missing_tokens: tuple[str, ...] = ("", "null", "none", "nan", "n/a"),
) -> dict[str, Any]:
    """Audit a JSON array or explicit top-level array key.

    Loads the JSON document in memory. JSONL and arbitrary nested source schemas
    require dedicated adapters; no assumptions about CrisisMMD layout are made.
    """
    with Path(path).open("r", encoding=encoding) as handle:
        payload = json.load(handle)
    if records_key is not None:
        if not isinstance(payload, dict) or records_key not in payload:
            raise ValueError("records_key must identify a top-level JSON field")
        payload = payload[records_key]
    if not isinstance(payload, list):
        raise ValueError("Expected a JSON array of record objects")
    return _audit_records(
        payload, id_column=id_column, text_column=text_column,
        label_columns=label_columns, required_columns=required_columns,
        missing_tokens=missing_tokens,
    )


### src/chainshield/data/manifest.py


In [ ]:
%%chainshield_file src/chainshield/data/manifest.py
"""Create provenance records in memory; never write or download files."""

from __future__ import annotations

from datetime import datetime, timezone
import hashlib
from pathlib import Path
from typing import Any


def sha256_file(path: str | Path) -> str:
    """Hash a local file using bounded memory."""
    source = Path(path)
    digest = hashlib.sha256()
    with source.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def build_file_manifest(
    path: str | Path,
    *,
    dataset_id: str,
    source_url: str,
    source_revision: str,
    license_status: str,
    relative_to: str | Path,
    acquired_at_utc: str | None = None,
) -> dict[str, Any]:
    """Describe one file with an explicit dataset root and acquisition metadata.

    observed_at_utc is the audit timestamp, not an inferred download timestamp.
    The resolved file must stay under the explicitly supplied root.
    """
    for name, value in (
        ("dataset_id", dataset_id),
        ("source_url", source_url),
        ("source_revision", source_revision),
        ("license_status", license_status),
    ):
        if not isinstance(value, str) or not value.strip():
            raise ValueError(f"{name} is required")
    source = Path(path).resolve(strict=True)
    root = Path(relative_to).resolve(strict=True)
    relative = source.relative_to(root)
    before = source.stat()
    checksum = sha256_file(source)
    after = source.stat()
    if (before.st_size, before.st_mtime_ns) != (after.st_size, after.st_mtime_ns):
        raise RuntimeError("File changed during checksum calculation")
    return {
        "schema_version": 1,
        "dataset_id": dataset_id,
        "file": relative.as_posix(),
        "bytes": after.st_size,
        "sha256": checksum,
        "source_url": source_url,
        "source_revision": source_revision,
        "license_status": license_status,
        "acquired_at_utc": acquired_at_utc,
        "observed_at_utc": datetime.now(timezone.utc).isoformat(),
    }


### src/chainshield/research/__init__.py


In [ ]:
%%chainshield_file src/chainshield/research/__init__.py
"""Revised capstone analysis and evidence helpers; no automatic model training."""


### src/chainshield/research/eda.py


In [ ]:
%%chainshield_file src/chainshield/research/eda.py
"""Training-focused exploratory analysis. No model fitting or source mutation."""
from __future__ import annotations
from collections import Counter,defaultdict
from pathlib import Path
from datetime import datetime,timezone
import csv,hashlib,html,json,math,re,statistics,unicodedata,os

DATASETS=("dataco","india_nepal","crisismmd")
STOPWORDS_FOR_DISPLAY={"a","an","the","and","or","is","are","was","were","to","of","in","on","for","with","at","by","from","this","that","it","as","be"}
def digest(path):
    value=hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda:stream.read(1024*1024),b""):value.update(block)
    return value.hexdigest()
def truth(value):
    return str(value).strip().casefold() in {"1","true"}
def rows(path):
    with Path(path).open(encoding="utf-8",newline="") as stream:return list(csv.DictReader(stream))
def quantile(values,fraction):
    values=sorted(values)
    if not values:return None
    position=(len(values)-1)*fraction
    lo=math.floor(position);hi=math.ceil(position)
    return float(values[lo]+(values[hi]-values[lo])*(position-lo))
def describe(values):
    values=[float(x) for x in values if x is not None and math.isfinite(float(x))]
    if not values:return {"count":0}
    q1,q3=quantile(values,.25),quantile(values,.75);spread=q3-q1
    return {"count":len(values),"mean":statistics.fmean(values),"std":statistics.pstdev(values),
            "min":min(values),"p25":q1,"median":quantile(values,.5),"p75":q3,
            "p95":quantile(values,.95),"p99":quantile(values,.99),"max":max(values),
            "zeros":sum(x==0 for x in values),"negative":sum(x<0 for x in values),
            "outside_tukey_fences":sum(x<q1-1.5*spread or x>q3+1.5*spread for x in values),
            "outlier_policy":"Descriptive flags only; no rows removed"}
def numeric(value):
    try:
        number=float(value)
        return number if math.isfinite(number) else None
    except (TypeError,ValueError):return None
def label_counts(values):
    counts=Counter(values);total=sum(counts.values())
    return [{"label":name,"count":count,"percent":100*count/total if total else 0}
            for name,count in sorted(counts.items())]
def minimal_clean(text):
    return " ".join(unicodedata.normalize("NFC",html.unescape(str(text or ""))).split())
def redact(text):
    text=re.sub(r"https?://\S+","[URL]",str(text))
    text=re.sub(r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b","[EMAIL]",text)
    text=re.sub(r"(?<!\w)@\w+","[USER]",text)
    return re.sub(r"\b\d{7,}\b","[LONG_NUMBER]",text)
def verified_bundle(directory):
    quality_path=directory/"quality.json"
    quality=json.loads(quality_path.read_text(encoding="utf-8"))
    for name,expected in quality["output_sha256"].items():
        path=(directory/name).resolve();path.relative_to(directory.resolve())
        if digest(path)!=expected:raise ValueError("Prepared input integrity mismatch: "+str(path))
    return quality
def correlation(a,b):
    pairs=[(x,y) for x,y in zip(a,b) if x is not None and y is not None]
    if len(pairs)<3:return {"records":len(pairs),"pearson_r":None}
    x,y=zip(*pairs);mx,my=statistics.fmean(x),statistics.fmean(y)
    numerator=sum((u-mx)*(v-my) for u,v in pairs)
    denominator=math.sqrt(sum((u-mx)**2 for u in x)*sum((v-my)**2 for v in y))
    return {"records":len(pairs),"pearson_r":numerator/denominator if denominator else None}
def text_profile(records,task,tokenizer=None,vocabulary=None,include_examples=False):
    if not records:raise ValueError("No eligible training records for "+task)
    if any(row.get("split")!="train" for row in records):raise ValueError("Text EDA accepts training records only")
    texts=[r["text"] for r in records]
    word_counts=[len(t.split()) for t in texts]
    regex_tokens=[re.findall(r"\w+|[^\w\s]",minimal_clean(t).casefold(),flags=re.UNICODE) for t in texts]
    lengths=[len(t) for t in regex_tokens]
    all_words=Counter();bigrams=Counter();trigrams=Counter();per_class=defaultdict(Counter)
    for row in records:
        words=re.findall(r"\b\w+\b",row["text"].casefold(),flags=re.UNICODE)
        all_words.update(words);per_class[row["label"]].update(words)
        bigrams.update(" ".join(words[i:i+2]) for i in range(max(0,len(words)-1)))
        trigrams.update(" ".join(words[i:i+3]) for i in range(max(0,len(words)-2)))
    source_counts=Counter(r["source"] for r in records)
    source_labels=Counter((r["source"],r["label"]) for r in records)
    result={"partition":"train","records":len(records),"groups":len({r["group_id"] for r in records}),
       "class_balance":label_counts(r["label"] for r in records),
       "source_or_event_counts":[{"source":k,"count":v} for k,v in sorted(source_counts.items())],
       "source_label_counts":[{"source":source,"label":label,"count":count} for (source,label),count in sorted(source_labels.items())],
       "lengths":{"characters":describe(map(len,texts)),"whitespace_words":describe(word_counts),"bilstm_regex_tokens":describe(lengths)},
       "length_histograms":{
           "characters":[{"value":key,"count":value} for key,value in sorted(Counter(map(len,texts)).items())],
           "whitespace_words":[{"value":key,"count":value} for key,value in sorted(Counter(word_counts).items())],
           "bilstm_regex_tokens":[{"value":key,"count":value} for key,value in sorted(Counter(lengths).items())]},
       "regex_length_coverage":[{"limit":limit,"above_limit":sum(x>limit for x in lengths),"fraction":sum(x>limit for x in lengths)/len(lengths)} for limit in (64,128,256)],
       "text_patterns":{"documents_with_url":sum(bool(re.search(r"https?://",t)) for t in texts),
                        "documents_with_mention":sum(bool(re.search(r"(?<!\w)@\w+",t)) for t in texts),
                        "documents_with_hashtag":sum("#" in t for t in texts),
                        "retweet_prefix":sum(bool(re.match(r"\s*RT\b",t)) for t in texts),
                        "non_ascii_documents":sum(any(ord(c)>127 for c in t) for t in texts)},
       "vocabulary_types_for_display":len(all_words),
       "top_words_raw":all_words.most_common(30),
       "top_words_display_filtered":[(w,c) for w,c in all_words.most_common() if w not in STOPWORDS_FOR_DISPLAY][:30],
       "top_bigrams":bigrams.most_common(30),"top_trigrams":trigrams.most_common(30),
       "top_words_by_class":{label:count.most_common(15) for label,count in sorted(per_class.items())},
       "display_filter_policy":"A small stopword list is used for comparative display only; it does not change the training pipeline and retains negations.",
       "interpretation":"Common terms may reflect event/source artifacts. Token/length summaries describe training inputs, not classifier quality."}
    if vocabulary is not None:
        total=sum(len(t) for t in regex_tokens);unknown=sum(token not in vocabulary for text in regex_tokens for token in text)
        result["inherited_bilstm_vocabulary"]={"size":len(vocabulary),"tokens":total,"unknown_tokens":unknown,
            "unknown_fraction":unknown/total if total else 0,
            "interpretation":"Retrospective coverage of the copied project-1 screening vocabulary; not a newly fitted vocabulary"}
    if tokenizer is not None:
        token_lengths=[]
        for start in range(0,len(texts),256):
            batch=tokenizer(texts[start:start+256],truncation=False,padding=False,add_special_tokens=True,return_attention_mask=False)
            token_lengths.extend(len(ids) for ids in batch["input_ids"])
        result["lengths"]["distilbert_tokens_including_special"]=describe(token_lengths)
        result["length_histograms"]["distilbert_tokens_including_special"]=[{"value":key,"count":value} for key,value in sorted(Counter(token_lengths).items())]
        result["distilbert_coverage"]=[{"limit":limit,"above_limit":sum(x>limit for x in token_lengths),
            "fraction":sum(x>limit for x in token_lengths)/len(token_lengths)} for limit in (64,128,256,512)]
        result["distilbert_truncation_by_class"]=[{"label":label,"records":sum(r["label"]==label for r in records),
            "above_128":sum(n>128 and r["label"]==label for r,n in zip(records,token_lengths))}
            for label in sorted({r["label"] for r in records})]
    if include_examples:
        chosen=sorted(records,key=lambda r:hashlib.sha256(r["record_id"].encode()).hexdigest())[:5]
        result["private_redacted_examples"]=[{"record_ref":hashlib.sha256(r["record_id"].encode()).hexdigest()[:16],
            "label":r["label"],"source":r["source"],"canonical_text":redact(r["text"][:600])} for r in chosen]
        result["example_warning"]="Automatic redaction is incomplete; manually review before sharing."
    return result
def write_csv(path,items):
    items=list(items)
    if not items:return
    with path.open("x",encoding="utf-8",newline="") as stream:
        writer=csv.DictWriter(stream,fieldnames=list(items[0]));writer.writeheader();writer.writerows(items)
def build_profiles(root,tokenizer_dir=None,include_examples=False,prepared_root=None,datasets=None):
    root=Path(root).resolve()
    prepared=Path(prepared_root).resolve() if prepared_root else root/"data/processed"
    prepared.relative_to(root)
    inherited=prepared==(root/"data/processed")
    selected=tuple(datasets) if datasets is not None else DATASETS
    if not selected or len(set(selected))!=len(selected) or set(selected)-set(DATASETS):raise ValueError("Invalid dataset selection")
    config={"ml":{}}
    if "dataco" in selected:
        config_file=root/"configs/eda/dataco.json"
        if not config_file.exists():config_file=root/"configs/training.json"
        config=json.loads(config_file.read_text())
    source={};quality={};canonical={}
    for dataset in selected:
        directory=prepared/dataset;quality[dataset]=verified_bundle(directory);canonical[dataset]=rows(directory/"canonical.csv")
        for name in ["canonical.csv","quality.json","split_manifest.csv","label_map.json"]:
            path=directory/name;source[path.relative_to(root).as_posix()]=digest(path)
    tokenizer=None
    if tokenizer_dir:
        tokenizer_dir=Path(tokenizer_dir).resolve();tokenizer_dir.relative_to(root)
        manifest=json.loads((tokenizer_dir/"manifest.json").read_text())
        if manifest["revision"]!="12040accade4e8a0f71eabdb258fecc2e7e948be":raise ValueError("Unexpected tokenizer revision")
        for item in manifest["files"]:
            if item["file"] in {"config.json","tokenizer.json","tokenizer_config.json","vocab.txt"} and digest(tokenizer_dir/item["file"])!=item["sha256"]:
                raise ValueError("Tokenizer asset integrity mismatch")
        os.environ["HF_HUB_OFFLINE"]="1";os.environ["TRANSFORMERS_OFFLINE"]="1";os.environ["HF_HUB_DISABLE_TELEMETRY"]="1"
        os.environ["HF_HOME"]=str(root/".tmp/huggingface")
        from transformers import AutoTokenizer
        tokenizer=AutoTokenizer.from_pretrained(str(tokenizer_dir),local_files_only=True,trust_remote_code=False)
        source[(tokenizer_dir/"manifest.json").relative_to(root).as_posix()]=digest(tokenizer_dir/"manifest.json")
    profile={"schema_version":1,"created_at_utc":datetime.now(timezone.utc).isoformat(),
             "analysis_kind":("Retrospective EDA of inherited prepared inputs" if inherited else "Prospective EDA of revised prepared inputs before model fitting"),
             "prepared_root":prepared.relative_to(root).as_posix(),
             "model_decision_partition":"eligible training records only",
             "source_sha256":source,"datasets":{},"text_tasks":{}}
    from importlib import import_module
    for dataset in selected:
        import_module("chainshield.research.eda_"+dataset).add_profiles(profile,canonical,prepared,tokenizer,include_examples,root,inherited,config)
    profile["preparation_controls"]={dataset:{"pipeline_version":quality[dataset].get("pipeline_version"),
          "invariants":quality[dataset].get("invariants"),"limitations":quality[dataset].get("limitations")}
          for dataset in selected}
    profile["normalization_demonstration"]={"kind":"synthetic, not an observed corpus record",
         "raw":"  Flood &amp; rescue\n assistance   is NOT available.  ","normalized":minimal_clean("  Flood &amp; rescue\n assistance   is NOT available.  ")}
    return profile
def export_profiles(profile,output):
    output=Path(output)
    if output.exists():raise FileExistsError("Do not overwrite an EDA run: "+str(output))
    output.mkdir(parents=True)
    (output/"dataset-profiles.json").write_text(json.dumps(profile,indent=2,ensure_ascii=False,allow_nan=False)+"\n",encoding="utf-8")
    data=profile["datasets"].get("dataco")
    balance=[]
    if data is not None:
        write_csv(output/"dataco-numeric-summary.csv",({"feature":k,**v} for k,v in data["numeric_summary"].items()))
        write_csv(output/"dataco-categorical-late-rates.csv",data["categorical_late_rates"])
        write_csv(output/"dataco-numeric-correlations.csv",data["numeric_correlations"])
        balance=[{"task":"dataco/shipment_late_risk",**r} for r in data["class_balance"]]
    for task,result in profile["text_tasks"].items():balance.extend({"task":task,**r} for r in result["class_balance"])
    write_csv(output/"training-class-balance.csv",balance)
    length=[{"task":task,"representation":representation,**stats}
            for task,result in profile["text_tasks"].items() for representation,stats in result["lengths"].items()]
    write_csv(output/"text-length-summary.csv",length)
    lines=["# Dataset EDA: data understanding and preparation","",profile["analysis_kind"],
           "Prepared input: "+profile.get("prepared_root","data/processed"),"Partition: eligible training records only.",
           "No models were fitted and no final-test scores were computed.",""]
    if data is not None:
        lines += ["## DataCo",f"Eligible training orders: {data['records']}.",
                  "Outliers are descriptive flags; no automatic removal.","",
                  "| Feature | Count | Median | p95 | Missing/invalid | Tukey flags |",
                  "|---|---:|---:|---:|---:|---:|"]
        for feature,stats in data["numeric_summary"].items():
            lines.append(f"| {feature} | {stats['count']} | {stats.get('median')} | {stats.get('p95')} | {stats['missing_or_invalid']} | {stats.get('outside_tukey_fences')} |")
    lines+=["","## Text tasks",""]
    for task,result in profile["text_tasks"].items():
        lines.extend(["### "+task,f"Eligible training rows: {result['records']}; groups: {result['groups']}.","",
            "| Label | Count | Percent |","|---|---:|---:|"])
        for row in result["class_balance"]:lines.append(f"| {row['label']} | {row['count']} | {row['percent']:.2f} |")
        for name,stats in result["lengths"].items():lines.append(f"- {name}: median {stats.get('median')}, p95 {stats.get('p95')}, p99 {stats.get('p99')}, max {stats.get('max')}.")
        for row in result.get("distilbert_coverage",[]):lines.append(f"- True DistilBERT length > {row['limit']}: {row['above_limit']} ({100*row['fraction']:.2f}%).")
        if "inherited_bilstm_vocabulary" in result:lines.append("- Existing BiLSTM vocabulary coverage is retrospective; see JSON.")
        lines.append("")
    lines+=["## Decisions pending","",
      "Review prediction-time feature availability, correlated feature groups, rare-class support, source/event effects and token coverage.",
      "Do not declare settings optimal, remove difficult labels or change final-test membership from these summaries alone.",
      "A revised preprocessing protocol and bounded comparisons require review before fresh training.","",
      "Charts are generated by the separate plotting step once its isolated dependencies are approved."]
    (output/"EDA_Review.md").write_text("\n".join(lines)+"\n",encoding="utf-8")
    (output/"completion.json").write_text(json.dumps({"created_at_utc":datetime.now(timezone.utc).isoformat(),
         "kind":"eda_statistics_only","model_training":False,"test_scoring":False,
         "file_sha256":{p.name:digest(p) for p in output.iterdir() if p.is_file()}},indent=2)+"\n",encoding="utf-8")

def __getattr__(name):
    if name=="dataco_profile":
        from .eda_dataco import dataco_profile
        return dataco_profile
    raise AttributeError(name)


### src/chainshield/research/plots.py


In [ ]:
%%chainshield_file src/chainshield/research/plots.py
"""Render integrated EDA evidence with matplotlib; no models or network use."""
from pathlib import Path
import json,os,html,math,re
def render(root,run):
    root=Path(root).resolve();run=Path(run).resolve()
    run.relative_to((root/"artifacts/research-v2/eda").resolve())
    data=json.loads((run/"dataset-profiles.json").read_text(encoding="utf-8"))
    os.environ["MPLCONFIGDIR"]=str(root/".tmp/matplotlib")
    Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True,exist_ok=True)
    os.environ["MPLBACKEND"]="Agg"
    import matplotlib
    matplotlib.use("Agg")
    from matplotlib import pyplot as plt
    import numpy as np
    from chainshield.research.eda import digest,rows,truth
    for name,expected in data["source_sha256"].items():
        if digest(root/name)!=expected:raise ValueError("EDA source changed: "+name)
    figures=run/"figures_v2"
    if figures.exists():raise FileExistsError("Do not overwrite rendered EDA evidence")
    figures.mkdir()
    plt.rcParams.update({"font.size":10,"axes.spines.top":False,"axes.spines.right":False,
                         "figure.facecolor":"#f7fafc","axes.facecolor":"white","savefig.facecolor":"#f7fafc"})
    panels=[]
    def save(fig,name,title,observation,decision):
        fig.text(.01,.005,"Training only | "+data.get("prepared_root","data/processed")+" | no final-test scoring",fontsize=8,color="#526477")
        fig.tight_layout(rect=(0,.035,1,.96))
        fig.savefig(figures/(name+".png"),dpi=160,bbox_inches="tight")
        fig.savefig(figures/(name+".svg"),bbox_inches="tight")
        plt.close(fig);panels.append({"name":name,"title":title,"observation":observation,"decision":decision})
    if "dataco" in data["datasets"]:
        from .plots_dataco import render_panels
        render_panels(data,root,save,plt,np)
    for index,(task,r) in enumerate(data["text_tasks"].items(),1):
        slug=task.replace("/","_");title=task.replace("_"," ")
        balance=r["class_balance"];fig,ax=plt.subplots(figsize=(12,max(4,len(balance)*.55)))
        bars=ax.barh([x["label"].replace("_"," ") for x in balance],[x["count"] for x in balance],color="#4878a8")
        ax.bar_label(bars,labels=[f'{x["count"]:,} ({x["percent"]:.2f}%)' for x in balance],padding=4,fontsize=9)
        ax.set_xlim(0,max(x["count"] for x in balance)*1.35);ax.invert_yaxis();ax.set_xlabel("Training records")
        ax.set_title(f'{title} | class support, n={r["records"]:,}')
        save(fig,"10_"+slug+"_classes",task+" class support",
             "Class support is reported per task; India-Nepal and CrisisMMD taxonomies remain separate.",
             "Retain rare labels, use macro/per-class metrics, and document support limitations instead of dropping difficult categories.")
        fig,ax=plt.subplots(figsize=(10,4))
        for representation,color in [("whitespace_words","#2b8cbe"),("distilbert_tokens_including_special","#e58b45")]:
            values=r.get("length_histograms",{}).get(representation)
            if not values:raise ValueError("Rerun EDA with length-frequency diagnostics before rendering")
            if values:
                ax.step([x["value"] for x in values],[x["count"] for x in values],where="mid",label=representation.replace("_"," "),color=color)
        ax.axvline(128,color="#a50f15",linestyle="--",label="Baseline limit 128")
        ax.set_xlabel("Words or tokenizer tokens per training record");ax.set_ylabel("Records")
        ax.set_title(f'{title} | actual length frequencies, n={r["records"]:,}');ax.legend()
        coverage=next(x for x in r["distilbert_coverage"] if x["limit"]==128)
        save(fig,"11_"+slug+"_lengths",task+" representation lengths",
             f'{coverage["above_limit"]:,}/{r["records"]:,} training records exceed 128 DistilBERT tokens.',
             "Keep 128 as the baseline when it covers the measured cohort. A longer context is not justified by these training lengths alone.")
        words=r["top_words_display_filtered"][:15];fig,ax=plt.subplots(figsize=(10,6))
        ax.barh([w for w,c in words],[c for w,c in words],color="#2b8cbe");ax.invert_yaxis();ax.set_xlabel("Training occurrences")
        ax.set_title(title+" | frequent terms; display-only stopword filter")
        save(fig,"12_"+slug+"_terms",task+" vocabulary inspection",
             "Frequent terms can reflect event names, URL fragments or source-specific wording.",
             "Inspect artifact sensitivity and compare conservative normalization. Display filtering is not the model preprocessing policy.")
        sources=r["source_or_event_counts"];fig,ax=plt.subplots(figsize=(11,4))
        bars=ax.barh([x["source"].replace("_"," ") for x in sources],[x["count"] for x in sources],color="#4878a8")
        ax.bar_label(bars,padding=3);ax.set_xlim(0,max(x["count"] for x in sources)*1.2);ax.invert_yaxis()
        ax.set_xlabel("Training records");ax.set_title(title+" | source/event composition")
        save(fig,"13_"+slug+"_sources",task+" source composition",
             "Source/event proportions are uneven; a mixed-source score can conceal transfer failures.",
             "Keep the source/event-held-out experiments and show class support alongside their scores.")
    cards=[]
    for panel in panels:
        cards.append("<section><h2>"+html.escape(panel["title"])+"</h2><img loading='lazy' src='figures_v2/"+panel["name"]+".png'>"
            +"<p><b>Observation:</b> "+html.escape(panel["observation"])+"</p><p><b>Decision:</b> "+html.escape(panel["decision"])+"</p></section>")
    content="<!doctype html><meta charset='utf-8'><title>ChainShield integrated data preparation review</title><style>body{font:16px/1.6 system-ui;color:#18354d;background:#edf3f8;margin:0}main{max-width:1150px;margin:30px auto;padding:20px}section{background:white;padding:25px;margin:25px 0;border-radius:12px}img{max-width:100%;height:auto}h1,h2{line-height:1.2}</style><main><h1>Data understanding | preparation decisions</h1><p>"+html.escape(data["analysis_kind"])+". Prepared input: "+html.escape(data.get("prepared_root","data/processed"))+". Training-only evidence; no final-test scoring or model fitting.</p>"+"".join(cards)+"</main>"
    (run/"Integrated_Data_Preparation_Review_v2.html").write_text(content,encoding="utf-8")
    record={"kind":"eda_visualizations","model_training":False,"test_scoring":False,"panels":panels,
            "source_profile_sha256":digest(run/"dataset-profiles.json"),
            "render_code_sha256":digest(root/"src/chainshield/research/plots.py"),
            "file_sha256":{p.relative_to(run).as_posix():digest(p) for p in figures.iterdir()}}
    record["file_sha256"]["Integrated_Data_Preparation_Review_v2.html"]=digest(run/"Integrated_Data_Preparation_Review_v2.html")
    (run/"visualization-completion-v2.json").write_text(json.dumps(record,indent=2)+"\n",encoding="utf-8")
    return len(panels)


### src/chainshield/research/preparation.py


In [ ]:
%%chainshield_file src/chainshield/research/preparation.py
"""Prospective preparation variants with fixed final-test membership; no model fitting."""
from __future__ import annotations
from pathlib import Path
from collections import defaultdict,Counter
from copy import deepcopy
from datetime import datetime,timezone
import csv,hashlib,io,json,re,html,unicodedata
from importlib import import_module

def sha(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda:stream.read(1024*1024),b""):h.update(block)
    return h.hexdigest()
def yes(value):return str(value).casefold() in {"1","true"}
def clean(text):return " ".join(unicodedata.normalize("NFC",html.unescape(str(text or ""))).split())
def normalize_urls_mentions(text):
    text=re.sub(r"https?://\S+"," urltoken ",clean(text),flags=re.IGNORECASE)
    return " ".join(re.sub(r"(?<!\w)@\w+"," usertoken ",text).split())
def key(text):return hashlib.sha256(clean(text).casefold().encode()).hexdigest()
def csv_rows(path):
    with Path(path).open(encoding="utf-8",newline="") as stream:return list(csv.DictReader(stream))
def canonical_scalar(value):
    if value is None:return ""
    return str(int(value)) if isinstance(value,bool) else str(value)
def records_digest(records,identity):
    columns=sorted(records[0])
    values=[[canonical_scalar(row[column]) for column in columns] for row in sorted(records,key=lambda r:canonical_scalar(r[identity]))]
    return hashlib.sha256(json.dumps({"columns":columns,"rows":values},separators=(",",":"),ensure_ascii=False).encode()).hexdigest()
def manifest_key(row):return (row.get("view",""),row["task"],row["record_id"])
def frozen_test_digest(manifest):
    entries=sorted((manifest_key(r),r["label"]) for r in manifest if r["split"]=="test" and yes(r["eligible"]))
    return hashlib.sha256(json.dumps(entries,separators=(",",":")).encode()).hexdigest()
def exclude(row,reason):
    row["eligible"]=False
    row["exclusion_reasons"]="|".join(sorted(set(filter(None,row.get("exclusion_reasons","").split("|")))|{reason}))

def validate_partition_groups(manifest):
    groups=defaultdict(set)
    for row in manifest:
        if yes(row["eligible"]):
            if row["split"] not in {"train","dev","test"} or not row["label"]:
                raise ValueError("Invalid eligible record")
            view=row.get("view","shared")
            task=row["task"] if view=="original_task" else "shared_tasks"
            groups[(view,task,row["group_id"])].add(row["split"])
    if any(len(values)>1 for values in groups.values()):raise ValueError("Eligible group crosses partitions")
    return True

def write_csv(path,items):
    if not items:raise ValueError("Empty output")
    columns=list(items[0])
    if any(set(row)!=set(columns) for row in items):raise ValueError("Inconsistent output fields")
    with path.open("x",encoding="utf-8",newline="") as stream:
        writer=csv.DictWriter(stream,fieldnames=columns);writer.writeheader()
        for row in items:writer.writerow({k:canonical_scalar(v) for k,v in row.items()})
def prepare_research(root,output_root,expected_reference=None,datasets=None,config_path=None):
    root=Path(root).resolve();output_root=Path(output_root).resolve()
    output_root.relative_to((root/"data/processed/research-v2").resolve())
    if output_root.exists():raise FileExistsError("Preserve prior prepared evidence: "+str(output_root))
    config_path=Path(config_path) if config_path else root/"configs/preprocessing.json"
    config=json.loads(config_path.read_text())
    modules={name:None for name in ("dataco","india_nepal","crisismmd")}
    selected=tuple(datasets) if datasets is not None else tuple(modules)
    if not selected or len(set(selected))!=len(selected) or set(selected)-set(modules):raise ValueError("Invalid dataset selection")
    modules={name:import_module("chainshield.data."+name) for name in selected}
    base={};references={};source={}
    identities={"dataco":"order_id","india_nepal":"record_id","crisismmd":"tweet_id"}
    for dataset,module in modules.items():
        acquisition_path=root/"data/manifests"/(dataset+".json")
        acquisition=json.loads(acquisition_path.read_text())
        if acquisition["status"]!="downloaded_and_integrity_verified":raise ValueError("Unverified raw inputs")
        for item in acquisition["files"]:
            path=(root/item["file"]).resolve();path.relative_to((root/"data/raw"/dataset).resolve())
            if path.stat().st_size!=item["bytes"] or sha(path)!=item["sha256"]:
                raise ValueError("Raw integrity mismatch")
        source[dataset]={"source_revision":acquisition["source_revision"],"manifest_sha256":sha(acquisition_path)}
        print("Fresh preparation from verified raw:",dataset,flush=True)
        bundle=module.prepare(root/"data/raw"/dataset,config)
        reference_directory=root/"data/processed"/dataset
        old_quality=json.loads((reference_directory/"quality.json").read_text())
        for filename,expected in old_quality["output_sha256"].items():
            if sha(reference_directory/filename)!=expected:raise ValueError("Reference bundle changed")
        reference=csv_rows(reference_directory/"canonical.csv")
        if records_digest(bundle["canonical"],identities[dataset])!=records_digest(reference,identities[dataset]):
            raise ValueError("Fresh base preparation differs from inherited canonical: "+dataset)
        old_manifest=csv_rows(reference_directory/"split_manifest.csv")
        if frozen_test_digest(bundle["split_manifest"])!=frozen_test_digest(old_manifest):
            raise ValueError("Fresh base preparation changed locked final-test membership")
        validate_partition_groups(bundle["split_manifest"])
        base[dataset]=bundle;references[dataset]={"canonical_semantic_sha256":records_digest(reference,identities[dataset]),
            "final_test_identity_label_sha256":frozen_test_digest(old_manifest),
            "split_manifest_sha256":sha(reference_directory/"split_manifest.csv")}
    if expected_reference is not None:
        for dataset,expected in expected_reference.items():
            if dataset not in references:continue
            for key,value in expected.items():
                if references[dataset].get(key)!=value:raise ValueError("Independent reproduction contract differs: "+dataset+"/"+key)
    if "crisismmd" in base:
        from .preparation_crisismmd import comparison_cohort,apply_variant
    cohort=comparison_cohort(base["crisismmd"]["canonical"],base["crisismmd"]["split_manifest"]) if "crisismmd" in base else None
    variants={}
    for name,normalized in (("control",False),("url_mentions",True)):
        variants[name]={dataset:deepcopy(bundle) for dataset,bundle in base.items() if dataset!="crisismmd"}
        if "crisismmd" in base:variants[name]["crisismmd"]=apply_variant(base["crisismmd"],cohort,normalized)
    # Both representations must use exactly the same eligible records and labels.
    def cohort_identity(bundle):
        return sorted((manifest_key(r),r["split"],r["label"],yes(r["eligible"])) for r in bundle["split_manifest"])
    if "crisismmd" in base and cohort_identity(variants["control"]["crisismmd"])!=cohort_identity(variants["url_mentions"]["crisismmd"]):
        raise ValueError("Representation candidates have different cohorts")
    # Validate every cohort before creating output directories.
    for variant,bundles in variants.items():
        for dataset,bundle in bundles.items():
            validate_partition_groups(bundle["split_manifest"])
            if frozen_test_digest(bundle["split_manifest"])!=references[dataset]["final_test_identity_label_sha256"]:
                raise ValueError("Preflight changed final-test identities")
            old_classes=defaultdict(set);new_classes=defaultdict(set)
            for row in base[dataset]["split_manifest"]:
                if row["split"]=="train" and yes(row["eligible"]):
                    old_classes[(row["task"],row.get("view",""))].add(row["label"])
            for row in bundle["split_manifest"]:
                if row["split"]=="train" and yes(row["eligible"]):
                    new_classes[(row["task"],row.get("view",""))].add(row["label"])
            if dict(old_classes)!=dict(new_classes):
                raise ValueError("Preparation would remove a training class")
    policy={"schema_version":1,"pipeline_version":"research-v2-preparation-2","control":"minimal inherited normalization",
            "candidate":"replace URL/mention spans with urltoken/usertoken; preserve negation, hashtag words and Unicode",
            "isolation":"latest existing partition retained; earlier overlapping rows excluded; never move held-out records into training",
            "training_duplicates":"one deterministic representative for identical training label; quarantine conflicting training labels",
            "evaluation_duplicates":"original held-out identities retained; report dependence/group support during evaluation",
            "base_preprocessing_config_sha256":sha(config_path),"reference":references,"sources":source,
            "model_training":False,"test_scoring":False}
    output_root.mkdir(parents=True)
    all_files={};summaries={}
    for variant,bundles in variants.items():
        summaries[variant]={}
        for dataset,bundle in bundles.items():
            directory=output_root/variant/dataset;directory.mkdir(parents=True)
            write_csv(directory/"canonical.csv",bundle["canonical"])
            write_csv(directory/"split_manifest.csv",bundle["split_manifest"])
            if "provenance" in bundle:write_csv(directory/"provenance.csv",bundle["provenance"])
            with (directory/"label_map.json").open("x",encoding="utf-8") as stream:
                json.dump(bundle["label_map"],stream,indent=2,ensure_ascii=False);stream.write("\n")
            if frozen_test_digest(bundle["split_manifest"])!=references[dataset]["final_test_identity_label_sha256"]:
                raise ValueError("Variant changed locked final-test identities")
            validate_partition_groups(bundle["split_manifest"])
            counts=defaultdict(Counter);train_labels=defaultdict(Counter);reasons=Counter()
            for row in bundle["split_manifest"]:
                task=row["task"]+"/"+row.get("view","default")
                if yes(row["eligible"]):
                    counts[task][row["split"]]+=1
                    if row["split"]=="train":train_labels[task][row["label"]]+=1
                else:reasons.update(filter(None,row.get("exclusion_reasons","").split("|")))
            # Every declared task must retain all labels previously present in its training cohort.
            original_classes=defaultdict(set)
            for row in base[dataset]["split_manifest"]:
                if row["split"]=="train" and yes(row["eligible"]):
                    original_classes[row["task"]+"/"+row.get("view","default")].add(row["label"])
            for task,labels in original_classes.items():
                if set(train_labels[task])!=labels:raise ValueError("Preparation lost a training class: "+task)
            quality={"schema_version":2,"pipeline_version":policy["pipeline_version"],"variant":variant,
                "canonical_records":len(bundle["canonical"]),"eligible_split_counts_by_task_view":{k:dict(v) for k,v in counts.items()},
                "eligible_training_label_counts":{k:dict(v) for k,v in train_labels.items()},
                "exclusion_reason_counts":dict(reasons),"source":source[dataset],
                "base_preprocessing_config_sha256":sha(config_path),"reference":references[dataset],
                "invariants":{"raw_integrity_verified":True,"fresh_base_matches_reference":True,
                    "final_test_identity_and_labels_preserved":True,"eligible_group_split_independence":True,
                    "training_classes_preserved":True,"candidate_cohorts_matched":True},
                "limitations":["Preparation does not establish model accuracy or operational benefit",
                    "Held-out labels are retained for integrity; no final-test metrics calculated",
                    "Normalized duplicate held-out identities remain to preserve the locked test cohort"],
                "prepared_at_utc":datetime.now(timezone.utc).isoformat()}
            for field in ("feature_columns","identifier_columns","excluded_predictors","prediction_cutoff"):
                if field in bundle["quality"]:quality[field]=bundle["quality"][field]
            quality["output_sha256"]={path.name:sha(path) for path in directory.iterdir() if path.is_file()}
            with (directory/"quality.json").open("x",encoding="utf-8") as stream:
                json.dump(quality,stream,indent=2);stream.write("\n")
            all_files.update({path.relative_to(output_root).as_posix():sha(path) for path in directory.iterdir() if path.is_file()})
            summaries[variant][dataset]={"eligible_counts":quality["eligible_split_counts_by_task_view"],
                "training_labels":quality["eligible_training_label_counts"],"exclusions":dict(reasons)}
    record={**policy,"created_at_utc":datetime.now(timezone.utc).isoformat(),"cohort_summaries":summaries,
            "file_sha256":all_files,
            "preparation_code_sha256":sha(root/"src/chainshield/research/preparation.py")}
    (output_root/"preparation-manifest.json").write_text(json.dumps(record,indent=2)+"\n",encoding="utf-8")
    (output_root/"completion.json").write_text(json.dumps({"manifest_sha256":sha(output_root/"preparation-manifest.json"),
          "model_training":False,"test_scoring":False},indent=2)+"\n",encoding="utf-8")
    return record

def __getattr__(name):
    if name in ("comparison_cohort","apply_variant"):
        return getattr(import_module("chainshield.research.preparation_crisismmd"),name)
    raise AttributeError(name)


### configs/research-v2/preparation.json


In [ ]:
%%chainshield_file configs/research-v2/preparation.json
{
  "schema_version": 1,
  "pipeline_version": "research-v2-preparation-2",
  "variants": [
    "control",
    "url_mentions"
  ],
  "output_root": "data/processed/research-v2/paired_v2",
  "normalization_candidate": "URL/mention placeholder normalization only",
  "comparison_cohort": "normalized structural isolation for both representations",
  "heldout_policy": "preserve original final-test identities and labels; never train on held-out records",
  "selection_status": "candidate comparison pending; no winner claimed",
  "view_policy": "original_task isolated per task; shared view isolated across tasks"
}


### configs/research-v2/reproduction-contract.json


In [ ]:
%%chainshield_file configs/research-v2/reproduction-contract.json
{
  "schema_version": 1,
  "role": "Expected semantic digests only; no local paths, data, model files or runtime dependency",
  "reference": {
    "dataco": {
      "canonical_semantic_sha256": "43fa2ad26ba100004b27b2f25e72da9a43500b58e649a4a959bb1d64c995f907",
      "final_test_identity_label_sha256": "78693994b459830a0266e652d67236ada04373e92d45c2c500b0c0bf970bce09"
    },
    "india_nepal": {
      "canonical_semantic_sha256": "8a6d24afa447facedf7c1f85736450ee03b45b2f1589e3a2a0ef990714734c0b",
      "final_test_identity_label_sha256": "3ef11364fa7ed34be4160ec45754e818310affd16883dbce15810009c76ae8ae"
    },
    "crisismmd": {
      "canonical_semantic_sha256": "a04fd48d48a84daaa1fe797320c01c6d5973f97149a36ad82c25e8d4840d203e",
      "final_test_identity_label_sha256": "d9fae0bad4b997650be1e9e00bbd11413f9de7989a8487d4434f167d6d5d765b"
    }
  }
}


### scripts/dataset_research_workflow.py


In [ ]:
%%chainshield_file scripts/dataset_research_workflow.py
"""Dataset-focused preparation/EDA stages; assembly verifies all three before modeling."""
from pathlib import Path
from datetime import datetime,timezone
import argparse,json,sys,shutil,importlib.util
ROOT=Path(__file__).resolve().parents[1]
sys.path.insert(0,str(ROOT/"src"))
from chainshield.research.preparation import prepare_research,sha,write_csv
DATASETS=("dataco","india_nepal","crisismmd")
def verify_stage(path):
    marker=json.loads((path/"completion.json").read_text())
    if sha(path/"preparation-manifest.json")!=marker["manifest_sha256"]:raise ValueError("Preparation manifest changed")
    record=json.loads((path/"preparation-manifest.json").read_text())
    for name,expected in record["file_sha256"].items():
        source=(path/name).resolve();source.relative_to(path.resolve())
        if sha(source)!=expected:raise ValueError("Prepared file changed")
    return record
def prepare_baseline(dataset):
    directory=ROOT/"data/processed"/dataset
    if (directory/"quality.json").exists():
        if not (ROOT/"docs"/("dataset-readiness-"+dataset+".md")).exists():raise ValueError("Partial baseline report; restore clean temporary runtime")
        quality=json.loads((directory/"quality.json").read_text())
        for name,digest in quality["output_sha256"].items():
            if sha(directory/name)!=digest:raise ValueError("Baseline preparation changed")
        return
    if directory.exists():raise FileExistsError("Partial baseline: restore a clean temporary runtime")
    from chainshield.research import readiness as helpers
    from importlib import import_module
    module=import_module("chainshield.data."+dataset)
    config_path=ROOT/"configs/preprocessing"/(dataset+".json");config=json.loads(config_path.read_text())
    manifest_path=ROOT/"data/manifests"/(dataset+".json");manifest=json.loads(manifest_path.read_text())
    if manifest["status"]!="downloaded_and_integrity_verified":raise ValueError("Raw acquisition incomplete")
    for item in manifest["files"]:
        source=(ROOT/item["file"]).resolve();source.relative_to((ROOT/"data/raw"/dataset).resolve())
        if source.stat().st_size!=item["bytes"] or sha(source)!=item["sha256"]:raise ValueError("Raw integrity mismatch")
    bundle=module.prepare(ROOT/"data/raw"/dataset,config)
    checks=helpers.validate_bundle(dataset,bundle)
    bundle["quality"].update({"pipeline_version":config["pipeline_version"],"preprocessing_config_sha256":sha(config_path),
        "source":{k:manifest[k] for k in ("source_revision","published_license")},"invariants":checks,
        "prepared_at_utc":datetime.now(timezone.utc).isoformat()})
    bundle["quality"]["source"]["manifest_sha256"]=sha(manifest_path)
    directory.mkdir(parents=True)
    for name,key in (("canonical.csv","canonical"),("split_manifest.csv","split_manifest"),("provenance.csv","provenance")):
        if key in bundle:helpers.write_csv(directory/name,bundle[key])
    (directory/"label_map.json").write_text(json.dumps(bundle["label_map"],indent=2,ensure_ascii=False)+"\n",encoding="utf-8")
    bundle["quality"]["output_sha256"]={p.name:sha(p) for p in directory.iterdir() if p.is_file()}
    (directory/"quality.json").write_text(json.dumps(bundle["quality"],indent=2,ensure_ascii=False)+"\n",encoding="utf-8")
    report=ROOT/"docs"/("dataset-readiness-"+dataset+".md");report.parent.mkdir(parents=True,exist_ok=True)
    report.write_text("# "+dataset+" readiness\n\nPrepared only this dataset from its verified public raw inputs.\n\n```json\n"+json.dumps(bundle["quality"],indent=2,ensure_ascii=False)+"\n```\n\nThese checks establish structural readiness, not predictive performance.\n\nModel training must fit transformations only on training data; final test partitions stay sealed.\n",encoding="utf-8")
def prepare_dataset(dataset):
    prepare_baseline(dataset)
    path=ROOT/"data/processed/research-v2/dataset_stages"/dataset
    if (path/"completion.json").exists():verify_stage(path);return
    contract=json.loads((ROOT/"configs/research-v2/reproduction-contract.json").read_text())["reference"]
    prepare_research(ROOT,path,expected_reference={dataset:contract[dataset]},datasets=(dataset,),config_path=ROOT/"configs/preprocessing"/(dataset+".json"))
def eda_dataset(dataset):
    from chainshield.research.eda import build_profiles,export_profiles
    from chainshield.research.plots import render
    tokenizer=None if dataset=="dataco" else ROOT/"artifacts/research-v2/tokenizer/distilbert"
    for variant in (("control","url_mentions") if dataset=="crisismmd" else ("control",)):
        output=ROOT/"artifacts/research-v2/eda"/(dataset+"_"+variant)
        prepared=ROOT/"data/processed/research-v2/dataset_stages"/dataset/variant
        if not (output/"completion.json").exists():
            export_profiles(build_profiles(ROOT,tokenizer,False,prepared,datasets=(dataset,)),output)
        if not (output/"visualization-completion-v2.json").exists():render(ROOT,output)
        print("Dataset EDA:",dataset,variant,flush=True)


def main():
    p=argparse.ArgumentParser();p.add_argument("action",choices=("prepare","eda"));p.add_argument("--dataset",required=True,choices=DATASETS);a=p.parse_args()
    if a.action=="prepare":prepare_dataset(a.dataset)
    else:eda_dataset(a.dataset)
if __name__=="__main__":main()


### src/chainshield/data/text_utils.py


In [ ]:
%%chainshield_file src/chainshield/data/text_utils.py
import hashlib,html,unicodedata

def clean_text(value):
    return " ".join(unicodedata.normalize("NFC", html.unescape(str(value or ""))).split())

def text_key(value):
    text = clean_text(value).casefold()
    return hashlib.sha256(text.encode("utf-8")).hexdigest() if text else ""


### src/chainshield/research/readiness.py


In [ ]:
%%chainshield_file src/chainshield/research/readiness.py
import csv

def validate_bundle(dataset, bundle):
    """Enforce invariants before writing any prepared output."""
    identity = "order_id" if dataset=="dataco" else "tweet_id" if dataset=="crisismmd" else "record_id"
    canonical = bundle["canonical"]
    if not canonical or len({r[identity] for r in canonical})!=len(canonical):
        raise ValueError("Canonical records must be nonempty and uniquely identified")
    groups = {}
    for row in bundle["split_manifest"]:
        if not row["eligible"]:
            continue
        if row["split"] not in {"train","dev","test"} or not row["label"]:
            raise ValueError("Eligible records require a valid split and label")
        view = row.get("view","shared")
        # Original per-task benchmarks may intentionally differ between tasks.
        key = (view,row["task"] if view=="original_task" else "all_tasks",row["group_id"])
        previous = groups.setdefault(key,row["split"])
        if previous!=row["split"]:
            raise ValueError("Eligible group crosses split boundaries")
    if dataset=="dataco":
        dates = {s:[r["order_date"] for r in canonical if r["eligible"] and r["split"]==s]
                 for s in ("train","dev","test")}
        if any(not values for values in dates.values()):
            raise ValueError("All chronological partitions must be nonempty")
        if not max(dates["train"])<min(dates["dev"]) or not max(dates["dev"])<min(dates["test"]):
            raise ValueError("DataCo chronological separation failed")
        forbidden = set(bundle["quality"]["excluded_predictors"])
        if forbidden.intersection(bundle["quality"]["feature_columns"]):
            raise ValueError("Disallowed model feature")
    return {"canonical_identity_unique":True,"eligible_group_split_independence":True,
            "chronology_checked":dataset=="dataco"}

def write_csv(path, rows):
    if not rows:
        raise ValueError("Refusing an empty CSV without an explicit schema")
    fields = list(rows[0])
    if any(set(row)!=set(fields) for row in rows):
        raise ValueError("Inconsistent output schema")
    with path.open("x",encoding="utf-8",newline="") as handle:
        writer = csv.DictWriter(handle,fieldnames=fields)
        writer.writeheader()
        for row in rows:
            writer.writerow({k:int(v) if isinstance(v,bool) else v for k,v in row.items()})


### src/chainshield/data/tokenizer_assets.py


In [ ]:
%%chainshield_file src/chainshield/data/tokenizer_assets.py
import hashlib,json
from pathlib import Path
from urllib.request import Request,urlopen
REPOSITORY="distilbert/distilbert-base-uncased"
REVISION="12040accade4e8a0f71eabdb258fecc2e7e948be"
FILES={
 "LICENSE":(11356,"f49a4e16e68b128803cc2dcea614603632b04eac","git"),
 "README.md":(8577,"1db487d688fd5ddc3c6714a4a3a3fe5f679cbf04","git"),
 "config.json":(483,"150367d8744161cd17b3f6462a14f3a9648752da","git"),
 "tokenizer.json":(466062,"949a6f013d67eb8a5b4b5b46026217b888021b88","git"),
 "tokenizer_config.json":(48,"e5c73d8a50df1f56fb5b0b8002d7cf4010afdccb","git"),
 "vocab.txt":(231508,"fb140275c155a9c7c5a3b3e0e77a9e839594a938","git"),
 "model.safetensors":(267954768,"5e3f1108e3cb34ee048634875d8482665b65ac713291a7e32396fb18f6ff0063","sha256"),
}
FILES={k:v for k,v in FILES.items() if k!="model.safetensors"}

def inspect_file(path,expected_size,expected_digest,kind):
    sha=hashlib.sha256()
    git=hashlib.sha1(f"blob {expected_size}\0".encode("ascii"))
    count=0
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda:handle.read(1048576),b""):
            sha.update(chunk)
            git.update(chunk)
            count+=len(chunk)
    if count!=expected_size or (sha.hexdigest() if kind=="sha256" else git.hexdigest())!=expected_digest:
        raise ValueError("Pretrained asset integrity failure: "+str(path))
    return {"file":Path(path).name,"bytes":count,"sha256":sha.hexdigest(),
            "upstream_digest":expected_digest,"upstream_digest_type":kind}

def ensure_pretrained(directory:Path):
    """Fetch only declared assets; existing assets are verified, never overwritten."""
    directory=Path(directory)
    directory.mkdir(parents=True,exist_ok=True)
    verified=[]
    for name,(size,digest,kind) in FILES.items():
        target=directory/name
        if not target.exists():
            partial=directory/(name+".partial")
            if partial.exists():
                raise FileExistsError("Interrupted download requires inspection before retry: "+str(partial))
            url=f"https://huggingface.co/{REPOSITORY}/resolve/{REVISION}/{name}"
            print("Downloading pinned pretrained asset: "+name,flush=True)
            with urlopen(Request(url,headers={"User-Agent":"ChainShield-neural/0.1",
                                             "Accept-Encoding":"identity"}),timeout=90) as response:
                if response.status!=200:
                    raise ValueError("Unexpected checkpoint download status")
                count=0
                progress=0
                with partial.open("xb") as handle:
                    while chunk:=response.read(1048576):
                        count+=len(chunk)
                        if count>size:
                            raise ValueError("Pretrained asset exceeds declared size")
                        handle.write(chunk)
                        if count-progress>=32*1048576:
                            print(f"  {name}: {count:,} / {size:,} bytes",flush=True)
                            progress=count
            inspect_file(partial,size,digest,kind)
            if target.exists():
                raise FileExistsError("Asset appeared during transfer")
            partial.rename(target)
        verified.append(inspect_file(target,size,digest,kind))
    manifest={"repository":REPOSITORY,"revision":REVISION,"license":"Apache-2.0",
              "files":verified,"trust_remote_code":False}
    path=directory/"manifest.json"
    if path.exists():
        if json.loads(path.read_text(encoding="utf-8"))!=manifest:
            raise ValueError("Pretrained manifest changed")
    else:
        with path.open("x",encoding="utf-8") as handle:
            json.dump(manifest,handle,indent=2)
            handle.write("\n")
    return manifest


### scripts/acquire_tokenizer_assets.py


In [ ]:
%%chainshield_file scripts/acquire_tokenizer_assets.py
from pathlib import Path
import sys
ROOT=Path(__file__).resolve().parents[1]
sys.path.insert(0,str(ROOT/"src"))
from chainshield.data.tokenizer_assets import ensure_pretrained
if __name__=="__main__":ensure_pretrained(ROOT/"artifacts/research-v2/tokenizer/distilbert")


In [ ]:
import hashlib,json
EXPECTED_STAGE_FILES={'.replication/controller.py': '8c092c59baffc0804aeca86b4f0ac3fb6d4be3cd96eea96c9e7299e559d9c9e6', 'src/chainshield/__init__.py': 'fd10d6903cafdd0b36dc065eaa94d2559bf3b7a5eafbc1fa9354f26403d6d575', 'src/chainshield/data/__init__.py': 'a8004a90802bb802bd808629c8048b39f3ca30a4271b39dbf85613637393a0f3', 'src/chainshield/data/audit.py': 'a457898b2fc3245d291fe33e477a573dd9a21eb9e686d1add193be7e1bd3c40b', 'src/chainshield/data/manifest.py': '7e9e61445a3e87655d696afae2a5a326a8ee6edb1dcd23ee47ef4fbb62041131', 'src/chainshield/research/__init__.py': '7a6bfc670027a2fe95322cd020bbbfb199e35bbd0b4b78dc70ed5cd7e0140121', 'src/chainshield/research/eda.py': '7b33a0a960bacb2e58bb320b5b93183c7c9fa17c8c2473273fde3a7c82cea6a1', 'src/chainshield/research/plots.py': '9b06ec2d094bc8d3357f77bb253b180ac213489153e2c7aa37727fcfff6ff38e', 'src/chainshield/research/preparation.py': '1a840365f261abcd2411eb533307cbb14a305d5264d7d3dc7163cad5568c1ab3', 'configs/research-v2/preparation.json': 'f385fef6d8f0fcee3d4045e9b7c6aa5af50cf0c587917b0d4e8569c1d3d3e692', 'configs/research-v2/reproduction-contract.json': '4f58dc46c9fa8d387c20d2e40011cdc376096637d4277a078891bdd23f43865b', 'scripts/dataset_research_workflow.py': 'b245fe0c13fd3bd8bbda2b924e00b36907383f427c470159e7cfc70a88cdf81a', 'src/chainshield/data/text_utils.py': '63a351d873e660548dcf322fca5d07951bc0dcf137050294f4368ed45a54c3b0', 'src/chainshield/research/readiness.py': '9cf2f4847439fd0226da05c2556f0eee538dd972b6bd75d37aa73164e73eded7', 'src/chainshield/data/tokenizer_assets.py': '8dc5437dd9d3aff0abfdb5e5681e137a8c9a733272216228af25fbaa8af56f70', 'scripts/acquire_tokenizer_assets.py': '873d1a38978eb34bfc8232019de5f3b98526dc6d8ec93cd3cb31668f8da91751'}
assert set(REGISTERED_FILES)==set(EXPECTED_STAGE_FILES)
for name,expected in EXPECTED_STAGE_FILES.items():
    if hashlib.sha256(REGISTERED_FILES[name].encode()).hexdigest()!=expected:raise ValueError("Changed source: "+name)


## 5. Attach, verify prerequisites and execute

Bootstrap connects Drive and restores committed work. Source installation commits only this stage definitions. Dependencies must already have completed; this notebook does not silently train them.

In [ ]:
# Connect the Colab experiment and load the shared runtime.
from pathlib import Path
CONTROLLER_SHA = '8c092c59baffc0804aeca86b4f0ac3fb6d4be3cd96eea96c9e7299e559d9c9e6'
IMPLEMENTATION_SHA256 = CONTROLLER_SHA
rt = None
if BOOTSTRAP_EXPERIMENT:
    if not ENABLE_DRIVE:
        raise PermissionError("Set ENABLE_DRIVE=True")
    runtime_code=REGISTERED_FILES[".replication/controller.py"]
    if hashlib.sha256(runtime_code.encode()).hexdigest()!=CONTROLLER_SHA:
        raise ValueError("Shared runtime version differs; do not mix notebook packages")
    exec(compile(runtime_code,"<Colab shared runtime>","exec"),globals())
    rt=Replica(EXPERIMENT_ID,drive_base=DRIVE_FOLDER,allow_drive=True)
    if CLEAR_STALE_WRITER_LOCK:
        rt.clear_stale_lock(acknowledge_no_other_writer=CONFIRM_NO_OTHER_WRITER)
else:
    print("Bootstrap disabled: no Drive access or filesystem changes")


In [ ]:
if RUN_STAGE:
    if rt is None:
        raise RuntimeError("Enable Drive/bootstrap first")
    rt.install_sources(REGISTERED_FILES)
else:
    print("Stage execution disabled")


### Create or reuse the managed CPU environment

This action executes only with RUN_STAGE=True. Progress and worker resource measurements are written to the experiment run directory.

In [ ]:
if RUN_STAGE:
    rt.environment(allow_install=ALLOW_PACKAGE_INSTALL)
else:
    print("Stage execution disabled")


## 6. Results and interpretation

Confirm the archive path and setup success. Package availability must be verified in the actual Colab runtime.

The following cell reads this experiment outputs; definitions alone are not evidence that a stage ran.

In [ ]:
if rt is None:
    print("No attached experiment")
else:
    print("Managed Python:",rt.python)
    print("Package freeze:",rt.control/"requirements-colab-resolved.txt")
    print("Persistent archive:",rt.archive)


## 7. Persist and hand over

Only after the preceding actions succeed, run the following cell to synchronize verified outputs and record this notebook completion. After an error, stop; repair/restart and rerun in order before recording completion. The next stage reads saved artifacts/components from Drive.

In [ ]:
if RUN_STAGE:
    rt.finish_stage('00')
else:
    print("Stage execution disabled")


## Recovery and boundaries

Run one writer per experiment. After interruption, disconnect and delete the Colab runtime, reconnect and restore this same experiment. Completed models are verified/reused; neural checkpoints resume compatible state. Unsynchronized work may repeat. Do not clear a writer lock until no other session is active.

No final-test fitting/scoring/tuning is performed. Model outputs do not verify incident truth or shipment exposure. Hosted Colab execution remains unverified until exercised in your session.